# 04 — Can we find the evidence in a complete PDF?

Notebook 03 compared readers **after giving them the correct evidence pages**.
A useful document-QA system must first find those pages. This notebook explains
and inspects a completed retrieval experiment. It reads verified public results;
running its cells does not download PDFs, launch GPUs, or repeat model inference.

## 1. Know which experiment is being measured

We freeze a multilingual **BM25 text-search baseline** and compare it with taking
pages in their original order. Every physical page is eligible. Rankings use only
the question and PDF text. Gold answers and evidence pages enter only afterward,
when the evaluator scores the saved rankings. No parameter was selected from these
results, and no test questions or labels were used.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.walkthrough import TABLE_STYLE, render_table
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger
from lava.retrieval.pipeline import load_public_report
from lava.retrieval.reporting import coverage_rows, document_rows, metric_rows, recall_chart

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.retrieval")
with logger.stage("01_load_verified_retrieval", heartbeat_seconds=15):
    summary = load_public_report(ROOT)
    display(
        HTML(
            TABLE_STYLE
            + render_table(
                [
                    {"Item": "Status", "Value": summary["status"]},
                    {"Item": "Labeled questions", "Value": summary["question_count"]},
                    {"Item": "PDFs", "Value": summary["document_count"]},
                    {
                        "Item": "Physical pages searched",
                        "Value": sum(row["page_count"] for row in summary["document_coverage"]),
                    },
                    {
                        "Item": "Reader answer score",
                        "Value": "Outside this component benchmark",
                    },
                ],
                caption="What has actually completed?",
            )
        )
    )

{"component": "notebook.retrieval", "elapsed_seconds": 0.0, "event": "01_load_verified_retrieval.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.911+00:00"}


Item,Value
Status,Full-document retrieval evaluated
Labeled questions,16
PDFs,5
Physical pages searched,74
Reader answer score,Outside this component benchmark


{"component": "notebook.retrieval", "elapsed_seconds": 0.005, "event": "01_load_verified_retrieval.completed", "level": "INFO", "stage_elapsed_seconds": 0.005, "timestamp_utc": "2026-09-07T04:39:16.916+00:00"}


## 2. Check extraction coverage before trusting any score

PyMuPDF extracts the native text layer from **every page**, in physical PDF order.
Textless pages stay in the index with no lexical signal; extraction failures stay
visible. A page can contain native text and still need visual understanding for a
chart or table. Native-text availability is not proof that all useful content was read.

This pilot contains all 16 supplied training labels: 15 Japanese and one Vietnamese
question. The five PDFs, not the 16 questions, are the more useful unit for checking
consistency. The single Vietnamese example cannot estimate language-wide performance.

In [2]:
with logger.stage("02_check_page_coverage", heartbeat_seconds=15):
    display(HTML(render_table(coverage_rows(summary), caption="No PDF pages silently dropped")))
    print(f"Questions with zero lexical signal: {summary['zero_signal_questions']}")

{"component": "notebook.retrieval", "elapsed_seconds": 0.01, "event": "02_check_page_coverage.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.921+00:00"}


PDF,All pages,Textless pages,Extraction errors,Gold pages without text
doc-01,18,0,0,0
doc-02,4,0,0,0
doc-03,15,0,0,0
doc-04,21,0,0,0
doc-05,16,0,0,0


Questions with zero lexical signal: 0
{"component": "notebook.retrieval", "elapsed_seconds": 0.012, "event": "02_check_page_coverage.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T04:39:16.923+00:00"}


## 3. Understand what the retriever does

Text is normalized with Unicode NFKC and case folding. We retain words plus
character bigrams and trigrams, so Japanese matching does not depend on spaces.
Vietnamese diacritics remain intact. BM25 rewards relevant term matches while
accounting for page length; repeated occurrences have diminishing returns.
Document frequencies come only from the PDF being searched. A page-number tie
break makes the ranking deterministic. Empty/no-match queries fall back to that
order and are counted as zero-signal cases.

This is an interpretable baseline, not a trained visual retriever. It establishes
the reference that a future multilingual embedding or visual method must improve.

In [3]:
with logger.stage("03_inspect_frozen_method", heartbeat_seconds=15):
    configuration = summary["implementation"]["config"]
    display(
        HTML(
            render_table(
                [
                    {"Setting": "BM25 k1", "Value": configuration["bm25"]["k1"]},
                    {"Setting": "BM25 b", "Value": configuration["bm25"]["b"]},
                    {
                        "Setting": "Page budgets",
                        "Value": ", ".join(map(str, configuration["budgets"])),
                    },
                    {"Setting": "OCR", "Value": "Not used in this baseline"},
                    {"Setting": "Hyperparameter tuning", "Value": "None"},
                ],
                caption="Settings chosen before looking at scores",
            )
        )
    )

{"component": "notebook.retrieval", "elapsed_seconds": 0.017, "event": "03_inspect_frozen_method.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.928+00:00"}


Setting,Value
BM25 k1,1.200
BM25 b,0.750
Page budgets,"1, 2, 3, 5, 10"
OCR,Not used in this baseline
Hyperparameter tuning,None


{"component": "notebook.retrieval", "elapsed_seconds": 0.019, "event": "03_inspect_frozen_method.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T04:39:16.930+00:00"}


## 4. Measure both partial and complete evidence

**Recall@k** is the fraction of gold evidence pages in the first k results.
**All-evidence@k** succeeds only when every required page is present. A multi-page
answer may fail even when recall looks high.

MRR@k rewards the first relevant hit. MAP@k rewards relevant pages occurring early;
this project's AP@k denominator is `min(number of relevant pages, k)`.
nDCG@k compares ranked relevance with an ideal ordering. All budgets were fixed
before this run. Report the entire curve rather than choosing k from the best-looking row.
A PDF with fewer than k pages contributes all its physical pages.

In [4]:
with logger.stage("04_measure_retrieval", heartbeat_seconds=15):
    display(HTML(recall_chart(summary)))
    display(HTML(recall_chart(summary, "all_evidence_at_k")))
    display(HTML(render_table(metric_rows(summary), caption="Question-weighted metrics")))

{"component": "notebook.retrieval", "elapsed_seconds": 0.023, "event": "04_measure_retrieval.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.934+00:00"}


Method,Pages (k),Evidence recall,All evidence found,MRR,MAP,nDCG
Page-order control,1,17.2%,12.5%,0.250,0.250,0.250
Page-order control,2,20.8%,12.5%,0.281,0.234,0.250
Page-order control,3,37.0%,25.0%,0.323,0.295,0.329
Page-order control,5,45.3%,37.5%,0.335,0.308,0.355
Page-order control,10,60.9%,56.2%,0.354,0.333,0.408
Multilingual BM25,1,53.6%,31.2%,0.812,0.812,0.812
Multilingual BM25,2,69.8%,56.2%,0.812,0.750,0.764
Multilingual BM25,3,83.9%,68.8%,0.854,0.771,0.812
Multilingual BM25,5,95.3%,87.5%,0.870,0.802,0.858
Multilingual BM25,10,100.0%,100.0%,0.870,0.820,0.878


{"component": "notebook.retrieval", "elapsed_seconds": 0.027, "event": "04_measure_retrieval.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-07T04:39:16.938+00:00"}


## 5. Check whether one PDF drives the result

A question average gives more weight to PDFs with more questions. The second
table gives each PDF equal weight. Large differences across PDFs are a reason
to investigate failure modes and expand document-isolated evaluation.
These fixed-model diagnostics are not cross-validation results or evidence of
generalization to the 200 hidden-label test PDFs.

In [5]:
with logger.stage("05_compare_documents", heartbeat_seconds=15):
    display(
        HTML(render_table(document_rows(summary), caption="BM25 complete-evidence coverage by PDF"))
    )
    display(
        HTML(
            render_table(metric_rows(summary, "document_average"), caption="Equal-document metrics")
        )
    )

{"component": "notebook.retrieval", "elapsed_seconds": 0.031, "event": "05_compare_documents.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.942+00:00"}


PDF,All evidence @ 1,All evidence @ 2,All evidence @ 3,All evidence @ 5,All evidence @ 10
doc-01,50.0%,75.0%,75.0%,75.0%,100.0%
doc-02,25.0%,50.0%,75.0%,100.0%,100.0%
doc-03,0.0%,0.0%,33.3%,100.0%,100.0%
doc-04,50.0%,100.0%,100.0%,100.0%,100.0%
doc-05,0.0%,0.0%,0.0%,0.0%,100.0%


Method,Pages (k),Evidence recall,All evidence found,MRR,MAP,nDCG
Page-order control,1,17.5%,10.0%,0.350,0.350,0.350
Page-order control,2,24.2%,10.0%,0.375,0.338,0.350
Page-order control,3,40.8%,20.0%,0.408,0.386,0.413
Page-order control,5,49.2%,31.7%,0.422,0.362,0.415
Page-order control,10,63.3%,48.3%,0.439,0.385,0.464
Multilingual BM25,1,47.5%,25.0%,0.817,0.817,0.817
Multilingual BM25,2,64.2%,45.0%,0.817,0.758,0.772
Multilingual BM25,3,80.8%,56.7%,0.856,0.781,0.818
Multilingual BM25,5,92.5%,75.0%,0.872,0.776,0.841
Multilingual BM25,10,100.0%,100.0%,0.872,0.816,0.878


{"component": "notebook.retrieval", "elapsed_seconds": 0.034, "event": "05_compare_documents.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T04:39:16.945+00:00"}


## 6. Verify persistence and elapsed time

The CPU runner verifies pinned source versions and hashes. Each completed PDF
extraction and question ranking is stored in S3 and read back before progress is
acknowledged. A fresh process restores these checkpoints. Conditional writes
protect completed work from another writer; corrupted or incompatible outputs
are rejected. Incomplete work is retried on the next run.

The current CPU process uses Studio. If Studio stops, its process stops; completed
checkpoints remain in S3 and the same command resumes them. Long GPU experiments
use independent SageMaker jobs. Completed analysis notebooks have separate checksum
manifests so viewing results does not require repeating either workload.

In [6]:
with logger.stage("06_verify_runtime_and_resume", heartbeat_seconds=15):
    validation = json.loads((ROOT / "reports/retrieval/validation.json").read_text())
    rows = [
        {
            "Run": row["run"],
            "Elapsed seconds": row["total_elapsed_seconds"],
            "PDFs computed": row["documents_computed"],
            "PDFs reused": row["documents_reused"],
            "Queries computed": row["queries_computed"],
            "Queries reused": row["queries_reused"],
        }
        for row in validation["attempts"]
    ]
    display(HTML(render_table(rows, caption="Measured first run and independent resume run")))

{"component": "notebook.retrieval", "elapsed_seconds": 0.039, "event": "06_verify_runtime_and_resume.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.950+00:00"}


Run,Elapsed seconds,PDFs computed,PDFs reused,Queries computed,Queries reused
Initial,8.548,5,0,16,0
Resume,0.898,0,5,0,16


{"component": "notebook.retrieval", "elapsed_seconds": 0.041, "event": "06_verify_runtime_and_resume.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T04:39:16.952+00:00"}


## 7. Conclusions from the completed retrieval benchmark

**BM25 improves evidence coverage over page order on these five PDFs.** At five retrieved pages, question-average evidence recall is 95.31% and all-evidence coverage is 87.50% (14/16 questions). Equal-document recall is 92.50% and complete coverage is 75.00%, showing how the question average can conceal document-level weaknesses. At ten pages all evidence is retrieved on this small pilot; that is not proof of perfect question answering.

The initial CPU evaluation took 8.548 seconds. A separate resumed process took 0.898 seconds and reused all five extractions and 16 rankings. This is direct evidence of working checkpoint recovery, alongside failure and corruption tests.

LAVA averages semantic answer credit and evidence-page F1. Retrieval recall, MAP, and nDCG cannot substitute for the answer score. The 9B score in Notebook 03 used oracle pages. This release evaluates retrieval and reading separately and claims no end-to-end answer score.

**This completes the retrieval deliverable and the portfolio benchmark.** Reader integration, visual retrieval, full test inference, and Kaggle submission are optional research directions, not requirements for this release.

References: [BM25 scoring](https://lucene.apache.org/core/9_12_1/core/org/apache/lucene/search/similarities/BM25Similarity.html),
[native PDF text extraction](https://pymupdf.readthedocs.io/en/latest/recipes-text.html),
[LAVA evaluation](https://lava-workshop.github.io/#evaluation).

In [7]:
with logger.stage("07_record_conclusions", heartbeat_seconds=15):
    display(
        HTML(
            render_table(
                [
                    {"Deliverable": "Oracle reader comparison", "Result": "Complete · 4B, 9B, 27B"},
                    {"Deliverable": "Full-document text retrieval", "Result": summary["status"]},
                    {
                        "Deliverable": "Integrated answer quality",
                        "Result": "Outside this release; no score claimed",
                    },
                    {
                        "Deliverable": "Kaggle submission",
                        "Result": "Optional; not required for this release",
                    },
                ],
                caption="Completed benchmark scope",
            )
        )
    )
logger.emit("retrieval.walkthrough.completed", question_count=summary["question_count"])

{"component": "notebook.retrieval", "elapsed_seconds": 0.046, "event": "07_record_conclusions.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T04:39:16.957+00:00"}


Deliverable,Result
Oracle reader comparison,"Complete · 4B, 9B, 27B"
Full-document text retrieval,Full-document retrieval evaluated
Integrated answer quality,Outside this release; no score claimed
Kaggle submission,Optional; not required for this release


{"component": "notebook.retrieval", "elapsed_seconds": 0.048, "event": "07_record_conclusions.completed", "level": "INFO", "stage_elapsed_seconds": 0.002, "timestamp_utc": "2026-09-07T04:39:16.958+00:00"}


{"component": "notebook.retrieval", "elapsed_seconds": 0.048, "event": "retrieval.walkthrough.completed", "level": "INFO", "question_count": 16, "timestamp_utc": "2026-09-07T04:39:16.959+00:00"}
